# Notebook 02 — INT8 Quantization Experiments

## What does this notebook do?

Runs **90 experiments**: 3 models × 6 calibration sizes × 5 seeds.

For each experiment:
1. Sample n images from ImageNet val set (calibration set)
2. Run calibration images through FP32 model → compute scale/zero-point
3. Convert FP32 → INT8
4. Evaluate INT8 model on validation set
5. Save all metrics to JSON

**Research question:** Is there a meaningful difference between n=10 and n=5000?

| Cell | What it does | Must re-run if session resets? |
|------|-------------|--------------------------------|
| Setup | Mount Drive, install packages | ✅ Always |
| ResNet-18 | 30 runs | Only if not saved to Drive yet |
| ResNet-50 | 30 runs | Only if not saved to Drive yet |
| MobileNetV2 | 30 runs | Only if not saved to Drive yet |
| Aggregate | Merge all JSONs → CSV | ✅ Always |

**Prerequisite:** Notebook 01 must be completed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os, sys

shutil.copytree('/content/drive/MyDrive/imagenet_v2',
                '/content/mlcalib_v2', dirs_exist_ok=True)
os.chdir('/content/mlcalib_v2')
sys.path.insert(0, '/content/mlcalib_v2')

!pip install datasets huggingface_hub scikit-learn matplotlib seaborn tqdm scipy -q

from huggingface_hub import login
login()

import torch
assert torch.cuda.is_available(), 'No GPU found! Go to Runtime > Change runtime type > T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')

# Set INT8 backend — required to avoid NotImplementedError during inference
torch.backends.quantized.engine = 'fbgemm'

from config import MODELS, CALIB_SIZES, SEEDS
total = len(MODELS) * len(CALIB_SIZES) * len(SEEDS)
print(f'\nTotal experiments: {len(MODELS)} models × {len(CALIB_SIZES)} sizes × {len(SEEDS)} seeds = {total} runs')
print(f'Models: {MODELS}')
print(f'Calibration sizes: {CALIB_SIZES}')
print(f'Seeds: {SEEDS}')

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


GPU: NVIDIA A100-SXM4-80GB

Total experiments: 4 models × 6 sizes × 5 seeds = 120 runs
Models: ['resnet18', 'resnet50', 'efficientnet_b0', 'mobilenet_v2']
Calibration sizes: [10, 50, 100, 500, 1000, 5000]
Seeds: [42, 123, 456, 789, 1024]


In [ ]:
import shutil, os

def save_to_drive(model_name=None):
    """Copy results/raw/ to Drive. If model_name given, prints how many files saved."""
    src = '/content/mlcalib_v2/results'
    dst = '/content/drive/MyDrive/imagenet_v2/results'
    shutil.copytree(src, dst, dirs_exist_ok=True)

    raw_dir = os.path.join(src, 'raw')
    if os.path.exists(raw_dir):
        files = [f for f in os.listdir(raw_dir) if f.endswith('.json')]
        if model_name:
            model_files = [f for f in files if f.startswith(model_name)]
            print(f'✓ Saved to Drive — {model_name}: {len(model_files)} runs backed up ({len(files)} total)')
        else:
            print(f'✓ Saved to Drive — {len(files)} total runs backed up')

def count_completed():
    """Show how many runs are already done per model."""
    raw_dir = '/content/mlcalib_v2/results/raw'
    if not os.path.exists(raw_dir):
        print('No completed runs yet.')
        return
    files = [f for f in os.listdir(raw_dir) if f.endswith('.json')]
    from config import MODELS
    print(f'Completed runs ({len(files)}/120 total):')
    for m in MODELS:
        n = len([f for f in files if f.startswith(m)])
        status = '✓ DONE' if n == 30 else f'{n}/30'
        print(f'  {m:<20} {status}')

count_completed()
print('\nHelper functions ready: save_to_drive(), count_completed()')

Completed runs (90/120 total):
  resnet18             ✓ DONE
  resnet50             ✓ DONE
  efficientnet_b0      0/30
  mobilenet_v2         ✓ DONE

Helper functions ready: save_to_drive(), count_completed()


In [ ]:
from src.experiment_runner import run_all_experiments

run_all_experiments(resume=True, models=['resnet18'])

save_to_drive('resnet18')
count_completed()

Models: ['resnet18']
Total runs planned: 30
Resume mode: 0 completed, 30 remaining.


Experiments:   0%|          | 0/30 [00:00<?, ?it/s]


[RUN] resnet18 | n=10 | seed=42
  Validation seti yükleniyor (5000 örnek)...


Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

  resnet18 yükleniyor (ImageNet pretrained)...
  ✓ resnet18 yüklendi
  FP32 accuracy: 69.06%


/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super().__init__(
Experiments: 100%|██████████| 30/30 [3:50:25<00:00, 460.86s/it]

  INT8 accuracy: 68.90% | ΔAcc: 0.1600% | ECE: 0.0329 | Size: 11.3MB | Latency: 383.4ms

✓ Completed: 30 | ✗ Failed: 0


✓ Saved to Drive — resnet18: 30 runs backed up (30 total)
Completed runs (30/120 total):
  resnet18             ✓ DONE
  resnet50             0/30
  efficientnet_b0      0/30
  mobilenet_v2         0/30


In [ ]:
from src.experiment_runner import run_all_experiments

run_all_experiments(resume=True, models=['resnet50'])

save_to_drive('resnet50')
count_completed()

Models: ['resnet50']
Total runs planned: 30
Resume mode: 0 completed, 30 remaining.


Experiments:   0%|          | 0/30 [00:00<?, ?it/s]


[RUN] resnet50 | n=10 | seed=42
  Validation seti yükleniyor (5000 örnek)...


Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

  resnet50 yükleniyor (ImageNet pretrained)...
  ✓ resnet50 yüklendi
  FP32 accuracy: 75.88%


/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super().__init__(
Experiments: 100%|██████████| 30/30 [4:44:05<00:00, 568.19s/it]

  INT8 accuracy: 75.22% | ΔAcc: 0.6600% | ECE: 0.0419 | Size: 25.0MB | Latency: 947.3ms

✓ Completed: 30 | ✗ Failed: 0


✓ Saved to Drive — resnet50: 30 runs backed up (60 total)
Completed runs (60/120 total):
  resnet18             ✓ DONE
  resnet50             ✓ DONE
  efficientnet_b0      0/30
  mobilenet_v2         0/30


In [ ]:
import os
drive_raw = '/content/drive/MyDrive/imagenet_v2/results/raw'
files = [f for f in os.listdir(drive_raw) if f.endswith('.json')]
print(f"Drive'da {len(files)} run kayıtlı")

from config import MODELS
for m in MODELS:
    n = len([f for f in files if f.startswith(m)])
    print(f"  {m}: {n}/30")

Drive'da 90 run kayıtlı
  resnet18: 30/30
  resnet50: 30/30
  efficientnet_b0: 0/30
  mobilenet_v2: 30/30


In [ ]:
save_to_drive()

✓ Saved to Drive — 90 total runs backed up


In [ ]:
from src.experiment_runner import run_all_experiments

run_all_experiments(resume=True, models=['efficientnet_b0'])

save_to_drive('efficientnet_b0')
count_completed()

Models: ['efficientnet_b0']
Total runs planned: 30
Resume mode: 0 completed, 30 remaining.


Experiments:   0%|          | 0/30 [00:00<?, ?it/s]


[RUN] efficientnet_b0 | n=10 | seed=42
  Validation seti yükleniyor (5000 örnek)...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Experiments:   3%|▎         | 1/30 [07:23<3:34:31, 443.85s/it]


KeyboardInterrupt: 

In [ ]:
save_to_drive()

In [ ]:
from src.experiment_runner import run_all_experiments

run_all_experiments(resume=True, models=['mobilenet_v2'])

save_to_drive('mobilenet_v2')
count_completed()

Models: ['mobilenet_v2']
Total runs planned: 30
Resume mode: 0 completed, 30 remaining.


Experiments:   0%|          | 0/30 [00:00<?, ?it/s]


[RUN] mobilenet_v2 | n=10 | seed=42
  Validation seti yükleniyor (5000 örnek)...


Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

  mobilenet_v2 yükleniyor (ImageNet pretrained)...
  ✓ mobilenet_v2 yüklendi
  FP32 accuracy: 71.00%


/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super().__init__(
/usr/local/lib/python3.12/dist-packages/torch/ao/nn/quantized/modules/functional_modules.py:294: UserWarning: must run observer before calling calculate_qparams.                                    Returning default scale and zero point 
  scale, zero_point = mod.activation_post_process.calculate_qparams()  # type: ignore[operator]
Experiments: 100%|██████████| 30/30 [3:51:48<00:00, 463.60s/it]

  INT8 accuracy: 33.06% | ΔAcc: 37.9400% | ECE: 0.2201 | Size: 4.0MB | Latency: 514.2ms

✓ Completed: 30 | ✗ Failed: 0


✓ Saved to Drive — mobilenet_v2: 30 runs backed up (90 total)
Completed runs (90/120 total):
  resnet18             ✓ DONE
  resnet50             ✓ DONE
  efficientnet_b0      0/30
  mobilenet_v2         ✓ DONE


In [ ]:
import os
drive_raw = '/content/drive/MyDrive/imagenet_v2/results/raw'
files = [f for f in os.listdir(drive_raw) if f.endswith('.json')]
print(f"Drive'da {len(files)} run kayıtlı")

from config import MODELS
for m in MODELS:
    n = len([f for f in files if f.startswith(m)])
    print(f"  {m}: {n}/30")

Drive'da 90 run kayıtlı
  resnet18: 30/30
  resnet50: 30/30
  efficientnet_b0: 0/30
  mobilenet_v2: 30/30


In [ ]:
save_to_drive()

✓ Saved to Drive — 90 total runs backed up


In [ ]:
from src.experiment_runner import aggregate_results

summary = aggregate_results()
print('\nSummary (first 12 rows):')
display(summary.head(12))

Summary saved: results/aggregated/summary.csv

Summary (first 12 rows):


,model,calib_size,fp32_accuracy_mean,fp32_accuracy_std,int8_accuracy_mean,int8_accuracy_std,accuracy_drop_mean,accuracy_drop_std,ece_mean,ece_std,model_size_mb_mean,model_size_mb_std,latency_ms_mean,latency_ms_std
0,mobilenet_v2,10,71.00,0.0,50.708,10.4981,20.292,10.4981,0.1280,0.0841,4.049,0.0,438.6038,27.2255
1,mobilenet_v2,50,71.00,0.0,35.212,3.3909,35.788,3.3909,0.2093,0.0436,4.049,0.0,456.1228,3.9530
2,mobilenet_v2,100,71.00,0.0,33.992,1.9174,37.008,1.9174,0.2225,0.0279,4.049,0.0,482.7794,20.3306
3,mobilenet_v2,500,71.00,0.0,35.884,2.6761,35.116,2.6761,0.1976,0.0397,4.049,0.0,490.6298,7.7714
4,mobilenet_v2,1000,71.00,0.0,34.568,2.5647,36.432,2.5647,0.2204,0.0251,4.049,0.0,485.8648,7.4543
5,mobilenet_v2,5000,71.00,0.0,34.868,1.3561,36.132,1.3561,0.2044,0.0274,4.049,0.0,509.3682,14.8676
6,resnet18,10,69.06,0.0,68.932,0.1180,0.128,0.1180,0.0332,0.0020,11.292,0.0,366.2220,7.4581
7,resnet18,50,69.06,0.0,68.784,0.1099,0.276,0.1099,0.0343,0.0022,11.292,0.0,362.9894,3.7958
8,resnet18,100,69.06,0.0,68.804,0.1292,0.256,0.1292,0.0347,0.0016,11.292,0.0,368.0158,3.3896
9,resnet18,500,69.06,0.0,68.856,0.1033,0.204,0.1033,0.0322,0.0014,11.292,0.0,368.2590,5.4504


In [ ]:
from config import MODELS

print('Validation checks (n=10 vs n=5000):')
print(f'{"Model":<20} {"n=10 ΔAcc":>10} {"n=5000 ΔAcc":>12} {"Diff":>8}')
print('-' * 55)
for model_name in MODELS:
    m = summary[summary['model'] == model_name]
    row_10   = m[m['calib_size'] == 10]
    row_5000 = m[m['calib_size'] == 5000]
    if not row_10.empty and not row_5000.empty:
        drop_10   = row_10['accuracy_drop_mean'].values[0]
        drop_5000 = row_5000['accuracy_drop_mean'].values[0]
        diff = drop_10 - drop_5000
        flag = '⚠️' if drop_5000 > 2.0 else '✓'
        print(f'{model_name:<20} {drop_10:>10.3f}% {drop_5000:>11.3f}% {diff:>7.3f}% {flag}')
    else:
        print(f'{model_name:<20}  (not enough data yet)')

Validation checks (n=10 vs n=5000):
Model                 n=10 ΔAcc  n=5000 ΔAcc     Diff
-------------------------------------------------------
resnet18                  0.128%       0.220%  -0.092% ✓
resnet50                  0.408%       0.344%   0.064% ✓
efficientnet_b0       (not enough data yet)
mobilenet_v2             20.292%      36.132% -15.840% ⚠️


In [ ]:
import shutil
for folder in ['results']:
    shutil.copytree(f'/content/mlcalib_v2/{folder}',
                    f'/content/drive/MyDrive/imagenet_v2/{folder}',
                    dirs_exist_ok=True)
print('✓ All results saved to Drive.')
count_completed()

✓ All results saved to Drive.
Completed runs (90/120 total):
  resnet18             ✓ DONE
  resnet50             ✓ DONE
  efficientnet_b0      0/30
  mobilenet_v2         ✓ DONE


In [ ]:
import time
from google.colab import runtime

print('waiting time for savings..')
time.sleep(300)

print('runtime unassigned')
runtime.unassign()